<a href="https://colab.research.google.com/github/muditkumar14/capstone-ames-data-intelligence/blob/main/Part4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# Part 4 - LLM Powered Feature
# Model Prediction Explanation Pipeline
# ============================================================

import json
import re
import requests
import joblib
import numpy as np
import pandas as pd

from google.colab import userdata

from jsonschema import (
    validate,
    ValidationError
)

print("=" * 60)
print("PART 4 - LLM POWERED FEATURE")
print("MODEL PREDICTION EXPLANATION PIPELINE")
print("=" * 60)

PART 4 - LLM POWERED FEATURE
MODEL PREDICTION EXPLANATION PIPELINE


In [4]:
# ============================================================
# Load Dataset
# ============================================================

print("\nLoading cleaned dataset...")

df = pd.read_csv("cleaned_data.csv")

print("Dataset loaded successfully.")
print(f"Dataset Shape : {df.shape}")

# ============================================================
# Load Best Model
# ============================================================

print("\nLoading best model...")

best_pipeline = joblib.load("best_model.pkl")
feature_columns = joblib.load("feature_columns.pkl")

print("Feature columns loaded successfully.")
print(f"Total Features : {len(feature_columns)}")
print("Best model loaded successfully.")

print(type(best_pipeline))


Loading cleaned dataset...
Dataset loaded successfully.
Dataset Shape : (2930, 82)

Loading best model...
Feature columns loaded successfully.
Total Features : 260
Best model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


In [5]:
# ============================================================
# OpenRouter Configuration
# ============================================================

print("=" * 60)
print("OPENROUTER CONFIGURATION")
print("=" * 60)

try:

    api_key = userdata.get("Openrouter")

    print("API Key loaded successfully.")

except Exception:

    api_key = None

    print("API Key not found.")

url = "https://openrouter.ai/api/v1/chat/completions"

model_name = "gpt-4.1-mini"

print(f"Model : {model_name}")

OPENROUTER CONFIGURATION
API Key loaded successfully.
Model : gpt-4.1-mini


In [6]:
# ============================================================
# Reusable LLM Function
# ============================================================

print("=" * 60)
print("CREATING LLM FUNCTION")
print("=" * 60)

def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=500
):

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    try:

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        response.raise_for_status()

        result = response.json()

        return result["choices"][0]["message"]["content"]

    except requests.exceptions.Timeout:

        print("Error: API request timed out.")
        return None

    except requests.exceptions.RequestException as e:

        print(f"Request Error: {e}")
        return None

    except (KeyError, IndexError):

        print("Unexpected API response:")
        print(result)
        return None

print("LLM Function created successfully.")

CREATING LLM FUNCTION
LLM Function created successfully.


In [7]:
# ============================================================
# Test OpenRouter Connection
# ============================================================

print("=" * 60)
print("TESTING API CONNECTION")
print("=" * 60)

system_prompt = (
    "You are a helpful assistant."
)

user_prompt = (
    "Reply with only the word hello."
)

response = call_llm(

    system_prompt=system_prompt,

    user_prompt=user_prompt,

    temperature=0.0

)

print("\nLLM Response\n")

print(response)

TESTING API CONNECTION

LLM Response

hello


In [8]:
# ============================================================
# PII Guardrail
# ============================================================

print("=" * 60)
print("PII GUARDRAIL")
print("=" * 60)

def contains_pii(text):

    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

    phone_pattern = (
        r"\b\d{10}\b"
        r"|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"
    )

    if re.search(email_pattern, text):
        return True

    if re.search(phone_pattern, text):
        return True

    return False

# Test Examples

example_1 = "My email is mudit@gmail.com"

example_2 = "The house has four bedrooms."

print("\nExample 1")

print(example_1)

print("Contains PII :", contains_pii(example_1))

print("\nExample 2")

print(example_2)

print("Contains PII :", contains_pii(example_2))

PII GUARDRAIL

Example 1
My email is mudit@gmail.com
Contains PII : True

Example 2
The house has four bedrooms.
Contains PII : False


In [9]:
# ============================================================
# JSON Schema
# ============================================================

print("=" * 60)
print("JSON SCHEMA")
print("=" * 60)

schema = {

    "type": "object",

    "properties": {

        "prediction_label": {
            "type": "string"
        },

        "confidence_level": {
            "type": "string"
        },

        "top_reason": {
            "type": "string"
        },

        "second_reason": {
            "type": "string"
        },

        "next_step": {
            "type": "string"
        }

    },

    "required": [

        "prediction_label",

        "confidence_level",

        "top_reason",

        "second_reason",

        "next_step"

    ]

}

print("JSON Schema created successfully.")

JSON SCHEMA
JSON Schema created successfully.


In [10]:
# ============================================================
# Prepare Features
# ============================================================

print("=" * 60)
print("PREPARE FEATURES")
print("=" * 60)

X = df.drop(columns=["SalePrice"])

# Ordinal Encoding
ordinal_mapping = {
    "Lot Shape": {
        "IR3": 0,
        "IR2": 1,
        "IR1": 2,
        "Reg": 3
    }
}

for col, mapping in ordinal_mapping.items():
    if col in X.columns:
        X[col] = X[col].map(mapping)

# One-Hot Encoding
nominal_cols = X.select_dtypes(include=["object", "category"]).columns

nominal_cols = [
    col
    for col in nominal_cols
    if col not in ordinal_mapping
]

X = pd.get_dummies(
    X,
    columns=nominal_cols,
    drop_first=True
)

# Match the training columns exactly
X = X.reindex(
    columns=feature_columns,
    fill_value=0
)

print("Feature preparation completed.")
print("Encoded Shape :", X.shape)

PREPARE FEATURES
Feature preparation completed.
Encoded Shape : (2930, 260)


In [11]:
# ============================================================
# Create Sample Houses
# ============================================================

print("=" * 60)
print("CREATE SAMPLE HOUSES")
print("=" * 60)

house_1 = X.iloc[[0]]
house_2 = X.iloc[[100]]
house_3 = X.iloc[[500]]

houses = {

    "House 1": house_1,

    "House 2": house_2,

    "House 3": house_3

}

print(f"Created {len(houses)} sample houses.")

CREATE SAMPLE HOUSES
Created 3 sample houses.


In [12]:
# ============================================================
# Generate Machine Learning Predictions
# ============================================================

print("=" * 60)
print("MODEL PREDICTIONS")
print("=" * 60)

prediction_results = {}

for house_name, house_data in houses.items():

    prediction = best_pipeline.predict(house_data)[0]

    probability = best_pipeline.predict_proba(house_data)[0]

    confidence = float(np.max(probability))

    probability_above = float(probability[1])

    prediction_label = (
        "Above Median Price"
        if prediction == 1
        else "Below Median Price"
    )

    if confidence >= 0.90:
        confidence_level = "High"
    elif confidence >= 0.75:
        confidence_level = "Medium"
    else:
        confidence_level = "Low"

    prediction_results[house_name] = {
        "prediction": int(prediction),
        "prediction_label": prediction_label,
        "confidence": round(confidence, 4),
        "confidence_level": confidence_level,
        "probability_above_median": round(probability_above, 4)
    }

prediction_df = pd.DataFrame(prediction_results).T

print(prediction_df)

MODEL PREDICTIONS
        prediction    prediction_label confidence confidence_level  \
House 1          1  Above Median Price        0.9             High   
House 2          1  Above Median Price       0.93             High   
House 3          1  Above Median Price        1.0             High   

        probability_above_median  
House 1                      0.9  
House 2                     0.93  
House 3                      1.0  


In [13]:
# ============================================================
# Generate LLM Explanations
# ============================================================

print("=" * 60)
print("LLM PREDICTION EXPLANATIONS")
print("=" * 60)

llm_outputs = {}

system_prompt = """
You are an expert machine learning assistant.

You explain machine learning predictions in simple language.

Return ONLY valid JSON.

The JSON format must be:

{
  "prediction_label": "",
  "confidence_level": "",
  "top_reason": "",
  "second_reason": "",
  "next_step": ""
}

Do not include markdown.
Do not include ```json.
Do not include explanations outside JSON.
"""

for house_name, result in prediction_results.items():

    # Get feature values for the current house
    features = houses[house_name].iloc[0]

    feature_text = ""

    for col, value in features.items():
        feature_text += f"{col}: {value}\n"

    user_prompt = f"""
The machine learning model predicted:

Prediction:
{result['prediction_label']}

Confidence Level:
{result['confidence_level']}

Probability of Above Median Price:
{result['probability_above_median']}

House Features:

{feature_text}

Explain why the model likely made this prediction.

Keep the explanation simple.

Return ONLY JSON.
"""

    response = call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=0.0
    )

    if response is None:
        print(f"{house_name}: API request failed.")
        continue

    # Remove markdown if present
    response = response.replace("```json", "")
    response = response.replace("```", "")
    response = response.strip()

    llm_outputs[house_name] = response

    print(f"\n{house_name}")
    print(response)

LLM PREDICTION EXPLANATIONS

House 1
{
  "prediction_label": "Above Median Price",
  "confidence_level": "High",
  "top_reason": "The house has a large lot area and a spacious living area, which are strong indicators of higher value.",
  "second_reason": "Good overall quality and condition, along with desirable features like two fireplaces, a two-car garage, and a paved driveway, contribute to the higher price prediction.",
  "next_step": "Consider verifying the neighborhood and recent sale conditions to confirm the market value and possibly get a professional appraisal."
}

House 2
{
  "prediction_label": "Above Median Price",
  "confidence_level": "High",
  "top_reason": "The house has a high overall quality (Overall Qual: 6) and good neighborhood (Somerst), which strongly influence higher prices.",
  "second_reason": "The house has a large living area (Gr Liv Area: 1573 sq ft) and a finished basement (BsmtFin SF 1: 378), adding value.",
  "next_step": "Consider verifying the conditi

In [14]:
# ============================================================
# Validate JSON Output
# ============================================================

print("=" * 60)
print("JSON VALIDATION")
print("=" * 60)

validated_outputs = {}

for house_name, response in llm_outputs.items():

    try:

        response = response.strip()
        parsed = json.loads(response)

        validate(
            instance=parsed,
            schema=schema
        )

        validated_outputs[house_name] = parsed

        print(f"{house_name} : JSON Valid")

    except json.JSONDecodeError as e:

        print(f"{house_name} : Invalid JSON")
        print(e)

    except ValidationError as e:

        print(f"{house_name} : Schema Validation Failed")
        print(e.message)

JSON VALIDATION
House 1 : JSON Valid
House 2 : JSON Valid
House 3 : JSON Valid


In [15]:
# ============================================================
# Temperature Comparison
# ============================================================

print("=" * 60)
print("TEMPERATURE COMPARISON")
print("=" * 60)

sample = prediction_results["House 1"]

sample_features = houses["House 1"].iloc[0]

feature_text = ""

for col, value in sample_features.items():
    feature_text += f"{col}: {value}\n"

system_prompt = """
You are an expert machine learning assistant.

Return ONLY valid JSON.

{
    "prediction_label":"",
    "confidence_level":"",
    "top_reason":"",
    "second_reason":"",
    "next_step":""
}
"""

user_prompt = f"""
The machine learning model predicted:

Prediction:
{sample['prediction_label']}

Confidence Level:
{sample['confidence_level']}

Probability of Above Median Price:
{sample['probability_above_median']}

House Features:

{feature_text}

Explain why the model made this prediction.

Return ONLY JSON.
"""

print("\nTemperature = 0.0\n")

response_temp0 = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.0
)

print(response_temp0)

print("\n" + "=" * 60)

print("\nTemperature = 0.7\n")

response_temp07 = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.7
)

print(response_temp07)

TEMPERATURE COMPARISON

Temperature = 0.0

```json
{
  "prediction_label": "Above Median Price",
  "confidence_level": "High",
  "top_reason": "The house has a high overall quality (Overall Qual: 6) and good basement condition with finished basement area, which increases its value.",
  "second_reason": "The house features desirable amenities such as 2 fireplaces, a 2-car garage, a large lot area (31770), and a wood deck, all contributing to a higher price prediction.",
  "next_step": "Review comparable sales in the Neighborhood_NAmes area to validate the model's prediction and consider any recent market trends that might affect the price."
}
```


Temperature = 0.7

```json
{
  "prediction_label": "Above Median Price",
  "confidence_level": "High",
  "top_reason": "The house has strong features such as a large lot area (31770 sq ft), a spacious first floor (1656 sq ft), two fireplaces, and a two-car garage with a large garage area (528 sq ft), which are positive indicators for higher p

In [16]:
# ============================================================
# Final Summary
# ============================================================

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary = []

for house_name, output in validated_outputs.items():

    summary.append({

        "House": house_name,

        "Prediction": output["prediction_label"],

        "Confidence": output["confidence_level"],

        "Top Reason": output["top_reason"],

        "Second Reason": output["second_reason"],

        "Next Step": output["next_step"]

    })

summary_df = pd.DataFrame(summary)

print(summary_df)

# Save summary
summary_df.to_csv(
    "prediction_summary.csv",
    index=False
)

print("\nPrediction summary saved successfully.")

print("\n")

print("=" * 60)
print("PART 4 COMPLETED SUCCESSFULLY")
print("=" * 60)

FINAL SUMMARY
     House          Prediction Confidence  \
0  House 1  Above Median Price       High   
1  House 2  Above Median Price       High   
2  House 3  Above Median Price       High   

                                          Top Reason  \
0  The house has a large lot area and a spacious ...   
1  The house has a high overall quality (Overall ...   
2  The house has high overall quality (8) and is ...   

                                       Second Reason  \
0  Good overall quality and condition, along with...   
1  The house has a large living area (Gr Liv Area...   
2  It has a large living area (2063 sq ft), a fin...   

                                           Next Step  
0  Consider verifying the neighborhood and recent...  
1  Consider verifying the condition of the house ...  
2  Consider comparing this house with similar hig...  

Prediction summary saved successfully.


PART 4 COMPLETED SUCCESSFULLY
